In [49]:
import pandas as pd
import re
import numpy as np


In [50]:
df = pd.read_csv(r"/content/train.csv", encoding='latin-1')

In [51]:
df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [52]:
df.dropna(axis=0,inplace=True)

In [53]:
X = df['text']
y = df['sentiment']

In [54]:
y = y.map({'positive':2,'negative':0,'neutral':1})

In [55]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [56]:
X.dropna(axis=0,inplace=True)

In [57]:
!pip install ftfy

In [58]:
import ftfy
X = X.apply(ftfy.fix_text)

In [59]:
!pip install nltk

In [60]:
import string
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
# from nltk.corpus import stopwords

lemmatizer = WordNetLemmatizer()
# stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lower casing
    text = text.lower()

    # 2. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # 3. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 4. Remove punctuations
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 5. Remove extra spaces
    text = text.strip()

    # 6. Tokenize
    tokens = word_tokenize(text)

    # 7. Remove stopwords
    #tokens = [word for word in tokens if word not in stop_words]

    # 8. Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens



In [61]:
import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [62]:
corpus = [preprocess_text(text) for text in X]

In [63]:
corpus[4]

['son',
 'of',
 'why',
 'couldnt',
 'they',
 'put',
 'them',
 'on',
 'the',
 'release',
 'we',
 'already',
 'bought']

In [64]:
!pip install gensim

In [65]:
from gensim.models import Word2Vec
model = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=2, workers=4)

In [66]:
import gensim.downloader as api
wv = api.load('word2vec-google-news-300')

In [67]:
import numpy as np

def document_vector(doc):
    doc = [word for word in doc if word in wv]  # keep only known words
    if len(doc) == 0:
        return np.zeros(300)  # if no known words
    return np.mean(wv[doc], axis=0)

In [68]:
X = np.array([document_vector(doc) for doc in corpus])

In [69]:
X_raw = df['text']

In [70]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X_raw,y,test_size=0.2,random_state=42)

In [71]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 20000
max_len = 200

tokenizer = Tokenizer(num_words=max_words,oov_token="OOV")
tokenizer.fit_on_texts(X_raw)

sequences = tokenizer.texts_to_sequences(X_raw)
X = pad_sequences(sequences,maxlen=max_len,padding='post')

In [72]:
X

array([[   2,  166,   20, ...,    0,    0,    0],
       [ 424,  120,    2, ...,    0,    0,    0],
       [   6, 1368,   10, ...,    0,    0,    0],
       ...,
       [ 236,   33,   12, ...,    0,    0,    0],
       [  21,    7,   29, ...,    0,    0,    0],
       [  30,   34, 6351, ...,    0,    0,    0]], dtype=int32)

In [73]:
embedding_dim = 300
word_index = tokenizer.word_index
embedding_matrix = np.zeros((max_words, embedding_dim))

for word, i in word_index.items():
    if i < max_words and word in wv:
        embedding_matrix[i] = wv[word]

In [74]:
embedding_matrix

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [-2.25585938e-01, -1.95312500e-02,  9.08203125e-02, ...,
         2.81982422e-02, -1.77734375e-01, -6.04248047e-03],
       ...,
       [-3.83300781e-02, -1.15722656e-01, -6.29882812e-02, ...,
        -3.76953125e-01,  1.89781189e-04, -4.73632812e-02],
       [-3.39355469e-02, -1.78710938e-01,  9.03320312e-02, ...,
        -5.07812500e-02,  1.72851562e-01,  2.94921875e-01],
       [-1.48925781e-02,  1.27929688e-01,  1.65039062e-01, ...,
         2.20703125e-01,  3.02734375e-01,  2.46093750e-01]])

In [75]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

In [76]:
my_model = Sequential()
my_model.add(Embedding(input_dim=max_words, output_dim=embedding_dim,weights=[embedding_matrix], trainable=False))
my_model.add(Bidirectional(LSTM(128,return_sequences=True)))
my_model.add(Dropout(0.5))
my_model.add(Bidirectional(LSTM(64)))
my_model.add(Dropout(0.5))
my_model.add(Dense(3,activation='softmax'))

In [77]:
# import tensorflow as tf

# loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True)
my_model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
history = my_model.fit(X,y,epochs=15,batch_size=64,validation_split=0.2)

Epoch 1/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 25s 54ms/step - accuracy: 0.5703 - loss: 0.8900 - val_accuracy: 0.7222 - val_loss: 0.6774
Epoch 2/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 17s 48ms/step - accuracy: 0.7173 - loss: 0.6770 - val_accuracy: 0.7329 - val_loss: 0.6541
Epoch 3/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 17s 50ms/step - accuracy: 0.7427 - loss: 0.6277 - val_accuracy: 0.7418 - val_loss: 0.6409
Epoch 4/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 20s 49ms/step - accuracy: 0.7549 - loss: 0.6053 - val_accuracy: 0.7458 - val_loss: 0.6285
Epoch 5/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 17s 49ms/step - accuracy: 0.7703 - loss: 0.5780 - val_accuracy: 0.7480 - val_loss: 0.6344
Epoch 6/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 17s 50ms/step - accuracy: 0.7749 - loss: 0.5678 - val_accuracy: 0.7549 - val_loss: 0.6121
Epoch 7/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 17s 49ms/step - accuracy: 0.7885 - loss: 0.5343 - val_accuracy: 0.7515 - val_loss: 0.6155
Epoch 8/15
344/344 ━━━━━━━━━━━━━━━━━━━━ 21s 51ms/step - accuracy: 0.8054 - loss: 0.4982 - 

In [78]:
def predict_sentiment(text, model, tokenizer, le, max_len=100):
    """
    Predict sentiment for a single text

    Args:
        text: Input text string
        model: Trained Keras model
        tokenizer: Fitted tokenizer
        le: Label encoder
        max_len: Maximum sequence length

    Returns:
        predicted_label: 'positive', 'negative', or 'neutral'
        probabilities: dict with confidence for each class
    """
    # Preprocess the text (same as training)
    text_clean = text.lower()
    text_clean = re.sub(r'<.*?>', '', text_clean)
    text_clean = re.sub(r'https?://\S+|www\.\S+', '', text_clean)

    # Tokenize and pad
    sequence = tokenizer.texts_to_sequences([text_clean])
    padded = pad_sequences(sequence, maxlen=max_len, padding='post')

    # Predict
    prediction = model.predict(padded, verbose=0)
    predicted_class = np.argmax(prediction[0])

    # Get label name
    predicted_label = le.inverse_transform([predicted_class])[0]

    # Get probabilities for all classes
    class_names = le.classes_
    probabilities = {class_names[i]: float(prediction[0][i]) for i in range(len(class_names))}

    return predicted_label, probabilities

# Example usage
test_texts = [
    "I love this movie! It's amazing!",
    "This is terrible, worst experience ever",
    "It was okay, nothing special",
    "movie was not great"
]

print("="*60)
for text in test_texts:
    label, probs = predict_sentiment(text, my_model, tokenizer, le)
    print(f"Text: '{text}'")
    print(f"Predicted: {label}")
    print(f"Probabilities:")
    for sentiment, prob in probs.items():
        print(f"  {sentiment}: {prob:.4f} ({prob*100:.2f}%)")
    print("-"*60)

Text: 'I love this movie! It's amazing!'
Predicted: 2
Probabilities:
  0: 0.0001 (0.01%)
  1: 0.0001 (0.01%)
  2: 0.9998 (99.98%)
------------------------------------------------------------
Text: 'This is terrible, worst experience ever'
Predicted: 0
Probabilities:
  0: 0.9932 (99.32%)
  1: 0.0064 (0.64%)
  2: 0.0003 (0.03%)
------------------------------------------------------------
Text: 'It was okay, nothing special'
Predicted: 1
Probabilities:
  0: 0.0272 (2.72%)
  1: 0.7321 (73.21%)
  2: 0.2406 (24.06%)
------------------------------------------------------------
Text: 'movie was not great'
Predicted: 0
Probabilities:
  0: 0.9725 (97.25%)
  1: 0.0157 (1.57%)
  2: 0.0117 (1.17%)
------------------------------------------------------------


In [79]:
import pickle
import json

# 1. Save the Keras model
model.save('sentiment_lstm_model.h5')  # or use .keras extension
# OR save in newer format
model.save('sentiment_lstm_model.keras')

# 2. Save the tokenizer
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

# 3. Save the label encoder
with open('label_encoder.pickle', 'wb') as handle:
    pickle.dump(le, handle, protocol=pickle.HIGHEST_PROTOCOL)

# 4. Save configuration (important!)
config = {
    'max_words': max_words,
    'max_len': max_len,
    'embedding_dim': embedding_dim
}

with open('model_config.json', 'w') as f:
    json.dump(config, f)

print("✓ Model saved successfully!")
print("Files created:")
print("  - sentiment_lstm_model.keras")
print("  - tokenizer.pickle")
print("  - label_encoder.pickle")
print("  - model_config.json")

✓ Model saved successfully!
Files created:
  - sentiment_lstm_model.keras
  - tokenizer.pickle
  - label_encoder.pickle
  - model_config.json
